In [28]:
import pandas as pd
df = pd.read_csv('../analys/databas/rådata/atellica analysdata/analyser-Tabell 1.csv',sep=';', quotechar='"')

df.columns = df.columns.str.replace(r'\s+', ' ', regex=True).str.strip()

#analyskod
df['analyskod'] = (df['prefix'] + df['analys']).str.lower().str.strip()
df = df.drop(columns=['analys'])

#prefix
df['prefix'] = df['prefix'].str.split('-').str[0].str.strip().str.lower()

#ackrediterad
df['ackrediterad'] = df['ackrediterad'].apply(lambda x: True if x == 'ja' else False)

#mätområde
df['mätområde'] = df['mätområde'].str.replace(',','.').str.replace('–','-').str.replace(':','',regex=False)
df[['min', 'max']] = df['mätområde'].str.split('-', expand=True)
df['mätområde'] = '[' + df['min'] + ',' + df['max'] + ']'

#larmvärde
df['larmvärde'] = df['larmvärde'].str.strip().str.lower().map({'ja':True, 'nej': False})


#omkörningar
def konvertera_intervall(varde, matomrade):
    if pd.isna(varde):
        return None
    
    v = varde.strip()
    v = v.replace(',','.')
    
    if v.lower() == 'hela området':
        return matomrade
    
    if v.startswith('\\leq') or v.startswith('≤'):
        tal = v.replace('\\leq', '').replace('≤','').strip()
        return f'(,{tal}]'
    
    if v.startswith('<'):
        tal = v.replace('<', '').strip()
        return f'(,{tal})'

    if v.startswith('>'):
        tal = v.replace('>', '').strip()
        return f'({tal},)'
    
    if v.startswith('\\geq') or v.startswith('≥'):
        tal = v.replace('\\geq', '').replace('≥','').strip()
        return f'[{tal},)'
    
    if '-' in v:
        del1, del2 = v.split('-')
        if '<' in del2.strip():
            return f'[{del1.strip()},{del2.strip().replace('<','')})'
        
        return f'[{del1.strip()},{del2.strip()}]'
    
    if '–' in v:
        del1, del2 = v.split('–')
        if '<' in del2.strip():
            return f'[{del1.strip()},{del2.strip().replace('<','')})'
        
        return f'[{del1.strip()},{del2.strip()}]'
    
    return v

df['omkörning_1'] = df.apply(
    lambda x: konvertera_intervall(x['omkörning_1'], x['mätområde']),
    axis=1
)

df['omkörning_2'] = df.apply(
    lambda x: konvertera_intervall(x['omkörning_2'], x['mätområde']),
    axis=1
)

df['omkörning_3'] = df.apply(
    lambda x: konvertera_intervall(x['omkörning_3'], x['mätområde']),
    axis=1
)

#omköningsavvikelse
def konvertera_avvikelse(avvikelse):
    if pd.isna(avvikelse):
        return (None,None)
    enhet = 'absolut'
    värde = None
    a = avvikelse.strip().replace(',','.')
    if '%' in a:
        enhet = 'procent'
        värde = a.split('%')[0].strip()
    else:
        värde = a
    return (värde,enhet)


df['omkörning_1_avvikelse'], df['omkörning_1_typ'] = zip(*df['omkörning_1_avvikelse'].apply(konvertera_avvikelse))
df['omkörning_2_avvikelse'], df['omkörning_2_typ'] = zip(*df['omkörning_2_avvikelse'].apply(konvertera_avvikelse))
df['omkörning_3_avvikelse'], df['omkörning_3_typ'] = zip(*df['omkörning_3_avvikelse'].apply(konvertera_avvikelse))

#H,I,L
df['hemolys_value'] = df['hemolys_value'].replace('saknas',None).replace('ej applicerbart',None)
df['hemolys_value'] = df['hemolys_value'].str.replace(':','',regex=False).str.replace(',','.')
df['ikteri_value'] = df['ikteri_value'].replace('saknas',None).replace('ej applicerbart',None)
df['ikteri_value'] = df['ikteri_value'].str.replace(':','',regex=False).str.replace(',','.')
df['lipemi_value'] = df['lipemi_value'].replace('saknas',None).replace('ej applicerbart',None)
df['lipemi_value'] = df['lipemi_value'].str.replace(':','',regex=False).str.replace(',','.')


#Skapa fil
df = df[['analyskod', 'prefix', 'extern_kod', 'ackrediterad',
       'process','mätområde', 'mätområde_comment', 'spädning','omkörning_okänd'
       ,'omkörning_1', 'omkörning_1_avvikelse','omkörning_1_typ','omkörning_2', 'omkörning_2_avvikelse','omkörning_2_typ',
       'omkörning_3', 'omkörning_3_avvikelse', 'omkörning_3_typ','hemolys_value',
       'hemolys_comment', 'ikteri_value', 'ikteri_comment', 'lipemi_value',
       'lipemi_comment', 'kalibrator_spårbarhet', 'specialregler']]

def rad_till_values(row):
    värden = []
    for v in row:
        if pd.isna(v):
            värden.append("NULL")
        else:
            text = str(v).replace("'", "''")
            värden.append(f"'{text}'")
    return f"({', '.join(värden)})"

alla_värden = df.apply(rad_till_values, axis=1)
kolumner = ", ".join(df.columns)

sql = f"INSERT INTO analyser ({kolumner}) VALUES\n" + ",\n".join(alla_värden) + ";"
sql.replace('–','-')


#labb, instrument, moduler
import re


df = pd.read_csv('../analys/databas/rådata/atellica analysdata/labb-Tabell 1.csv',sep=';', quotechar='"')
labs = ['LU', 'MA', 'MISSBR','HG','YS','LA','TR','BN II','KD','HM','ÄN','LU.1','MA.1']
df['sammanslagen'] = df[['LU', 'MA', 'MISSBR','HG','YS','LA','TR','BN II','KD','HM','ÄN','LU.1','MA.1']].apply(
    lambda row: ', '.join(row.dropna().astype(str)), axis=1
)

df = df.drop(columns=['LU', 'MA', 'MISSBR','HG','YS','LA','TR','BN II','KD','HM','ÄN','LU.1','MA.1'])
df['Modul'] = df['Modul'].replace({'e':'Cobas Pro','IM': 'Atellica-CH','CH': 'Atellica-IM'})
def parse_row(row):
    delar = []
    for lab in row['sammanslagen'].split(','):
        lab = lab.strip()
        analyskod = row['analyskod']
        modul = row['Modul']
        if modul == 'BNII':
            delar.append(f"('{analyskod}', 'LU', 'BN II')")
        else:
            delar.append(f"('{analyskod}', '{lab}', '{modul}')")
    return ',\n'.join(delar)

rader = []
for _, row in df.iterrows():
    rader.append(parse_row(row))

analysinnehav = 'INSERT INTO analysinnehav(analyskod, labkod, instrumentnamn) VALUES\n' + ',\n'.join(rader) + ';\n'

#medicinskt ansvariga, metodansvariga
df = pd.read_csv('../analys/databas/rådata/atellica analysdata/Människor-Tabell 1.csv',sep=';',quotechar='"')

df = df[['analyskod','Metodansvarig','Medicinskt ansvarig']]
df.columns = ['analyskod','metodansvarig','medicinskt ansvarig']
p = set()
df['metodansvarig'].apply(lambda x: p.add(str(x)))
df['medicinskt ansvarig'].apply(lambda x: p.add(str(x)))
p.remove('nan')

people = 'INSERT INTO användare (användarkod, namn) VALUES \n' 
usernames = {}
for person in p:
    parts = person.split(' ')
    username = f'{parts[0].lower()}-{parts[1][0].lower()}'
    people += f"('{username}','{person}'),\n"
    usernames[person] = username

people = people[:-2] + ';\n'

people += 'INSERT INTO metodansvariga (analyskod,användarkod) VALUES \n'
for _, row in df.iterrows():
    användarkod = usernames.get(row['metodansvarig'])
    if pd.isna(användarkod):
        continue
    people += f"('{row['analyskod']}','{användarkod}'),\n"
people = people[:-2] + ';\n'

people += 'INSERT INTO medicinskt_ansvariga (analyskod,användarkod) VALUES \n'
for _, row in df.iterrows():
    användarkod = usernames.get(row['medicinskt ansvarig'])
    if pd.isna(användarkod):
        continue
    people += f"('{row['analyskod']}','{användarkod}'),\n"
people = people[:-2] + ';\n'


#interna kontroller
df = pd.read_csv('../analys/databas/rådata/atellica analysdata/Kontroller-Tabell 1.csv',sep=';',quotechar='"')
df['Kontroll 3'] = df['Kontroll 3'].apply(lambda x: str(int(x)) if not pd.isna(x) else x)


kontroller = {}
for _,row in df.iterrows():
    if not pd.isna(row['Kontroll 1']) and not 'saknas' in row['Kontroll 1']:
        kontroller[str(row['Kontroll 1'])] = str(row['Kontrollnamn'])
    if not pd.isna(row['Kontroll 2']) and not 'saknas' in row['Kontroll 2']:
        kontroller[str(row['Kontroll 2'])] = str(row['Kontrollnamn.1'])
    if not pd.isna(row['Kontroll 3']) and not 'saknas' in str(row['Kontroll 3']):
        kontroller[str(row['Kontroll 3'])] = str(row['Kontrollnamn.2'])

kontroll_string = 'INSERT INTO kontroller (kontrollkod, namn) VALUES \n' 

for k,v in kontroller.items():
    kontroll_string += f"('{k}','{v}'),\n"
kontroll_string = kontroll_string[:-2] + ';\nINSERT INTO analyskontroller(kontrollkod,analyskod,kontrollnummer) VALUES\n'

for _,row in df.iterrows():
    if not pd.isna(row['Kontroll 1']) and not 'saknas' in row['Kontroll 1']:
        kontroll_string += f"('{row['Kontroll 1']}','{row['analyskod']}',1),\n"
    if not pd.isna(row['Kontroll 2']) and not 'saknas' in row['Kontroll 2']:
        kontroll_string += f"('{row['Kontroll 2']}','{row['analyskod']}',2),\n"
    if not pd.isna(row['Kontroll 3'])  and not 'saknas' in str(row['Kontroll 3']):
        kontroll_string += f"('{row['Kontroll 3']}','{row['analyskod']}',3),\n"

kontroll_string = kontroll_string[:-2] + ';\n'


#externa kontroller
ext_kontroll = set()
externa_kontroller = "INSERT INTO externa_kontroller (extern_kontrollkod, namn) VALUES \n"
for _,row in df.iterrows():
    if 'Externkontroll saknas' in str(row['Extern kontroll']):
        continue
    kontrollnamn = row['Extern kontroll']
    if kontrollnamn in ext_kontroll:
        continue
    ext_kontroll.add(kontrollnamn)
    externa_kontroller += f"('{kontrollnamn}','{kontrollnamn}'),\n"
externa_kontroller = externa_kontroller[:-2] +';\nINSERT INTO externa_kontroller_länk (extern_kontrollkod,analyskod) VALUES\n'

for _,row in df.iterrows():
    if row['Extern kontroll'] is None or pd.isna(row['Extern kontroll']) or 'Externkontroll saknas' in str(row['Extern kontroll']):
        continue
    externa_kontroller += f"('{row['Extern kontroll']}','{row['analyskod']}'),\n"
externa_kontroller = externa_kontroller[:-2] +';\n'


# #referensintervall och autovalidering
df = pd.read_csv('../analys/databas/rådata/atellica analysdata/resultat-Tabell 1.csv',sep=';',quotechar='"')
# referensintervall = 'INSERT INTO referensintervall (analyskod, kommentar,ålder,män,kvinnor) VALUES \n' 

# for _,row in df.iterrows():
#     referensintervall += f"('{row['analyskod']}','{row['Referensintervall']}','(,)','True','True'),\n"

# referensintervall = referensintervall[:-2] + ';\n'


# df['Autoval'] = df['Autoval'].str.replace(',','.')

# import re

def är_intervall(s):
    return bool(re.fullmatch(r'\d+\.?\d*\s*-\s*\d+\.?\d*', s))

autovalidering = 'INSERT INTO autovalidering (analyskod,ålder,intervall,kommentar) VALUES \n' 
for _,row in df.iterrows():
    if är_intervall(str(row['autoval'])):
        parts = row['autoval'].split('-')
        autovalidering += f"('{row['analyskod']}','(,)','[{parts[0].strip()},{parts[1].strip()}]',Null),\n"
    else:
         autovalidering += f"('{row['analyskod']}','(,)','(,)','{row['autoval']}'),\n"


autovalidering = autovalidering[:-2] + ';\n'

df = df[['analyskod','svarsgräns','svarsformat','enhet','adm_kortnamn','larmvärde']]

df['larmvärde'] = df['larmvärde'].str.strip().str.lower().map({'ja':True, 'nej': False}).fillna(False)

resultat = 'INSERT INTO resultat(svarsgräns,svarsformat,enhet,adm_kortnamn,larmvärde) VALUES\n'
for _,row in df.iterrows():
    resultat += f"('{row['svarsgräns']}', '{row['svarsformat']}', '{row['enhet']}', '{row['adm_kortnamn']}', '{row['larmvärde']}'),\n"
resultat = resultat[:-2] + ';\n'

link = ""

for i,row in df.iterrows():
    link += f"UPDATE ANALYSER SET resultatkod = {i+1} WHERE analyskod = '{row['analyskod']}';\n"


In [29]:
manual_input = ''
with open("../analys/databas/insert-manual-data.sql") as f:
    manual_input =f.read()

with open("../analys/databas/real-data.sql", "w") as f:
    f.write(sql)
    f.write("\n")
    f.write(manual_input)
    f.write(people)
    f.write(analysinnehav)
    f.write(kontroll_string)
    f.write(externa_kontroller)
    f.write(autovalidering)
    f.write(resultat)
    
